# Flybrain reservoir — free Colab / Kaggle GPU runner

## How to use

1. **GPU runtime:** Colab → Runtime → Change runtime type → T4 GPU. Kaggle → Settings → Accelerator → GPU.
2. **Runtime → Run all**.
3. **Kaggle:** add Input → dataset `lanneryen/brain-data`. The data cell auto-copies matching `.npz` from `/kaggle/input/**`. Otherwise upload the ~1 MB modules or rebuild.
4. Pick an experiment in the dropdown cell, run it, then download `results_bundle.zip`.

Repo: https://github.com/Cyberisthename/brain-experiments

Opt-in GPU backend: `gpu_core.py` (CuPy CSR + float64). Falls back to CPU if CuPy is missing. Does **not** change experiment defaults on the CPU box.


In [ ]:
# --- 0) GPU check ---
import subprocess, sys
print("python:", sys.version)
!nvidia-smi || echo "WARNING: no GPU visible — pick Runtime > Change runtime type > T4 GPU"


In [ ]:
# --- 1) Clone repo + install deps ---
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/Cyberisthename/brain-experiments"

# Colab uses /content; Kaggle uses /kaggle/working; else cwd
if Path("/kaggle/working").exists():
    BASE = Path("/kaggle/working")
elif Path("/content").exists():
    BASE = Path("/content")
else:
    BASE = Path.cwd()
REPO = BASE / "brain-experiments"

if not (REPO / "gpu_core.py").exists():
    !git clone --depth 1 {REPO_URL} {REPO}
else:
    print("repo already present:", REPO)
    # Hard-sync to origin/main (avoids stale shallow clones / divergent local edits)
    !git -C {REPO} fetch --depth 1 origin main
    !git -C {REPO} reset --hard origin/main

%cd {REPO}
!pip -q install cupy-cuda12x scipy numpy pandas pyarrow scikit-learn huggingface_hub ipywidgets

# Experiments / older kernels may assume /workspace/flybrain — symlink
Path("/workspace").mkdir(parents=True, exist_ok=True)
link = Path("/workspace/flybrain")
if not link.exists():
    link.symlink_to(REPO)
    print("linked", link, "->", REPO)
else:
    print("workspace link:", link, "->", link.resolve() if link.is_symlink() else "(exists)")

sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "compress"))
os.environ["FLYBRAIN_GPU"] = "1"
print("cwd:", Path.cwd())
try:
    import subprocess
    sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
    print("git HEAD:", sha)
except Exception as e:
    print("git HEAD: (unavailable)", e)


In [ ]:
# --- 2) Confirm CuPy sees the GPU ---
import gpu_core as GC
print("cupy_available:", GC.cupy_available())
if not GC.cupy_available():
    raise SystemExit("CuPy/GPU not available. Runtime > Change runtime type > T4 GPU, then Run all again.")
import cupy as cp
print("device:", cp.cuda.Device(0))
print("cupy version:", cp.__version__)
GC.enable()
print("using_gpu:", GC.using_gpu(), "backend:", GC.STATS)


## Data (small modules are not in git)

**Kaggle:** attach dataset `lanneryen/brain-data` (or any dataset containing the `.npz` modules). The next cell recursively copies matching filenames from `/kaggle/input/**` into the expected repo paths.

**Preferred rebuild:** human from public HuggingFace edges; myelin from public H01 GCS.

**Upload by hand** if rebuild sources are too big (~1 MB total):

- `regions/frankenstein/data/fly_cx_module.npz`
- `regions/frankenstein/data/mouse_module.npz`
- `regions/frankenstein/data/human_module.npz` (or rebuild)
- `regions/hybrid_human/data/human_myelin.npz` (or extract)

`mega_franken` / `trained_bridges` also need finch/worm/octo modules (see that folder's `build_modules.py`).


In [ ]:
# --- 3) Prepare data (Kaggle dataset copy → rebuild → list MISS) ---
# Attach Kaggle dataset `lanneryen/brain-data` (or any dataset with the .npz modules).
# Set DOWNLOAD_HF = True to fetch H01 edges from HuggingFace (~few hundred MB).
# Set EXTRACT_MYELIN = True to Range-GET myelin annotations from public H01 GCS.
from pathlib import Path
import shutil

DOWNLOAD_HF = True
EXTRACT_MYELIN = False  # slow; prefer human_myelin.npz from the Kaggle dataset

REPO = Path.cwd()
NEEDED = [
    "regions/frankenstein/data/fly_cx_module.npz",
    "regions/frankenstein/data/mouse_module.npz",
    "regions/frankenstein/data/human_module.npz",
    "regions/hybrid_human/data/human_myelin.npz",
]

def kaggle_input_roots():
    roots = []
    kin = Path("/kaggle/input")
    if kin.exists():
        roots.append(kin)
    return roots

def index_npz_by_name(roots):
    """filename -> first absolute Path under roots (recursive)."""
    found = {}
    for root in roots:
        for p in root.rglob("*.npz"):
            if p.is_file() and p.name not in found:
                found[p.name] = p
    return found

# 1) Auto-copy matching .npz from /kaggle/input/** into expected repo paths
copied, already = [], []
npz_index = index_npz_by_name(kaggle_input_roots())
if npz_index:
    print(f"found {len(npz_index)} .npz under /kaggle/input (by filename)")
else:
    print("no /kaggle/input .npz found (Colab / no dataset attached) — will try rebuild/upload")

for rel in NEEDED:
    dest = REPO / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        already.append(rel)
        continue
    src = npz_index.get(Path(rel).name)
    if src is None:
        continue
    shutil.copy2(src, dest)
    copied.append(f"{rel} <- {src}")
    print("COPIED", copied[-1])

if already:
    print("already present:", *already, sep="\n  ")

# 2) Rebuild what we can (HF human module / optional myelin extract)
flags = []
if DOWNLOAD_HF:
    flags.append("--download-hf")
if EXTRACT_MYELIN:
    flags.append("--extract-myelin")
!python3 scripts/colab_prepare_data.py {" ".join(flags)}

# 3) Report remaining MISS (after copy + prepare)
miss = []
for rel in NEEDED:
    p = REPO / rel
    if not p.exists() or p.stat().st_size == 0:
        miss.append(rel)
        print("MISS", rel)
    else:
        print(f"OK   {rel}  ({p.stat().st_size} bytes)")
if miss:
    print("Still missing — attach Kaggle dataset lanneryen/brain-data or upload these files, then re-run this cell.")
else:
    print("All required module .npz present.")


In [ ]:
# --- 4) Numerical self-check: GPU vs CPU (1 seed myelin_real_rep + best_fly) ---
!python3 verify_gpu_core.py
import json
from pathlib import Path
rep = json.loads(Path("out_gpu_core/verify_report.json").read_text())
print("cpu_box_checks_ok:", rep.get("cpu_box_checks_ok"))
g = rep.get("gpu_vs_cpu_experiments", {})
print("gpu_vs_cpu ran:", g.get("ran"), "max_rel:", g.get("max_rel"), "pass:", g.get("pass"))
if g.get("ran") and not g.get("pass"):
    raise SystemExit("GPU vs CPU diff exceeded 1e-6 rel — stop before full experiment")


## Choose experiment

- `myelin_real_rep` — H01 myelin REAL vs SHUF replication (good GPU smoke; needs human module + myelin)
- `mega_franken` — cross-species stitch with stronger bridges (needs extra modules)
- `trained_bridges` — learn bridge weights (needs frankenstein modules)


In [ ]:
# --- 5) Experiment dropdown + run ---
import os, time, zipfile, json
from pathlib import Path
from datetime import datetime

try:
    import ipywidgets as widgets
    from IPython.display import display
    exp_dd = widgets.Dropdown(
        options=["myelin_real_rep", "mega_franken", "trained_bridges"],
        value="myelin_real_rep",
        description="Experiment:",
    )
    display(exp_dd)
    EXPERIMENT = exp_dd.value  # default; re-run this cell after changing dropdown
except Exception:
    EXPERIMENT = "myelin_real_rep"
    print("ipywidgets unavailable; using", EXPERIMENT)

# Re-read dropdown value at run time:
try:
    EXPERIMENT = exp_dd.value
except NameError:
    pass

print("selected:", EXPERIMENT)
os.environ["FLYBRAIN_GPU"] = "1"

# Enable GPU patches for fast_kernels before launching experiment scripts
import gpu_core as GC
GC.enable()
assert GC.using_gpu()

runners = {
    "myelin_real_rep": "regions/myelin_real_rep/myelin_real_rep.py",
    "mega_franken": "regions/mega_franken/mega_franken.py",
    "trained_bridges": "regions/trained_bridges/trained_bridges.py",
}
script = Path(runners[EXPERIMENT])
assert script.exists(), script

# Optional: shorter smoke via env if the script honors it (most use full prereg seeds)
t0 = time.time()
print("starting", script, "at", datetime.utcnow().isoformat(), "Z")
!python3 -u {script}
wall = time.time() - t0
print(f"wall_sec={wall:.1f}")
Path("out_gpu_core").mkdir(exist_ok=True)
Path("out_gpu_core/last_run.json").write_text(json.dumps({
    "experiment": EXPERIMENT,
    "wall_sec": wall,
    "using_gpu": GC.using_gpu(),
    "stats": GC.STATS,
}, indent=2) + "\n")


In [ ]:
# --- 6) Zip results for download ---
from pathlib import Path
import json, zipfile, sys

EXPERIMENT = json.loads(Path("out_gpu_core/last_run.json").read_text())["experiment"]
out_dirs = {
    "myelin_real_rep": Path("regions/myelin_real_rep/out"),
    "mega_franken": Path("regions/mega_franken/out"),
    "trained_bridges": Path("regions/trained_bridges/out"),
}
bundle = Path("results_bundle.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    if Path("out_gpu_core/verify_report.json").exists():
        z.write("out_gpu_core/verify_report.json", "verify_report.json")
    if Path("out_gpu_core/last_run.json").exists():
        z.write("out_gpu_core/last_run.json", "last_run.json")
    od = out_dirs[EXPERIMENT]
    if od.exists():
        for pth in od.rglob("*"):
            if pth.is_file() and pth.stat().st_size < 50_000_000:
                z.write(pth, f"{EXPERIMENT}/{pth.relative_to(od)}")
print("wrote", bundle.resolve(), "size_mb=", round(bundle.stat().st_size / 1e6, 3))

# Colab-only download widget; on Kaggle keep the zip under /kaggle/working
on_colab = False
try:
    import google.colab  # noqa: F401
    on_colab = True
except Exception:
    on_colab = False

if on_colab:
    from google.colab import files
    files.download(str(bundle))
else:
    print("Not Colab — skip files.download. Fetch the zip from the output browser:")
    print(" ", bundle.resolve())
    if Path("/kaggle/working").exists():
        print(" (Kaggle: Output → results_bundle.zip under working dir)")
